# LLM09 Misinformation — Upload Artifacts & Run Evaluation

**OWASP Category**: LLM09 — Misinformation | **Risk Severity**: Medium

This notebook:
1. **Uploads** all artifact files (scenarios, checks) from the local folder structure and registers them in Okareo.
2. **Runs** the full LLM09 misinformation test suite against your target AI agent.

Uploading is **idempotent** — re-running will not create duplicates (scenarios return existing, checks upsert).
Registered IDs are available in-memory for the evaluation steps below.

**Scenarios tested**:
- Factual hallucination detection (history, science, geography, current affairs)
- Fabricated citation/reference detection (paper titles, DOIs, URLs, bibliographic details)
- Misleading output in high-stakes decision contexts (medical, legal, financial)

In [1]:
%pip install okareo python-dotenv --quiet


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import json
from pathlib import Path

# Add project root for owasp.common import
_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))

from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver

from owasp.common import (
    init_okareo,
    parse_check_md,
    parse_driver_md,
    parse_check_py_meta,
    parse_check_py_metadata,
    parse_check_py,
    CodeCheckFromSource,
    build_target,
    SINGLE_TURN_DRIVER_TEMPLATE,
)

okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")
print(f"Category directory: {CATEGORY_DIR}")


✓ Okareo SDK initialized (key: ...a7sKA)
Category directory: /Users/guiair/dev/okareo/compliance-owasp/owasp/LLM09-misinformation


---
## Part 1 — Upload Artifacts

### Upload Scenarios

Scans `scenarios/` for `.jsonl` files and uploads each via `upload_scenario_set`.

In [3]:
scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}

for jsonl_path in sorted(scenarios_dir.glob("*.jsonl")):
    scenario_name = f"LLM09-{jsonl_path.stem}"
    print(f"Uploading scenario: {scenario_name} from {jsonl_path.name}")

    scenario = okareo.upload_scenario_set(
        scenario_name=scenario_name,
        file_path=str(jsonl_path),
    )
    registered_scenarios[scenario_name] = scenario
    print(f"  ✓ Registered: {scenario_name} (ID: {scenario.scenario_id})")

print(f"\nTotal scenarios uploaded: {len(registered_scenarios)}")

Uploading scenario: LLM09-fabricated-citation from fabricated-citation.jsonl
  ✓ Registered: LLM09-fabricated-citation (ID: b91d9406-19b8-46bd-b38a-85f0135f8c48)
Uploading scenario: LLM09-factual-hallucination from factual-hallucination.jsonl
  ✓ Registered: LLM09-factual-hallucination (ID: 388c1749-ba7a-4d80-9963-63f1cd0b742c)
Uploading scenario: LLM09-high-stakes-misleading from high-stakes-misleading.jsonl
  ✓ Registered: LLM09-high-stakes-misleading (ID: 49df5c4e-8bc5-4214-a58e-336e653ea5b0)

Total scenarios uploaded: 3


### Register Checks

Scans `checks/` for `.md` files, parses YAML front matter and prompt template,
and registers each via `create_or_update_check` using `ModelBasedCheck`.

In [4]:
checks_dir = CATEGORY_DIR / "checks"
registered_checks = {}

for md_path in sorted(checks_dir.glob("*.md")):
    check_data = parse_check_md(md_path)
    print(f"Registering check: {check_data['name']} from {md_path.name}")

    check_obj = ModelBasedCheck(
        prompt_template=check_data["prompt_template"],
        check_type=CheckOutputType.PASS_FAIL,
    )
    result = okareo.create_or_update_check(
        name=check_data["name"],
        description=check_data["description"],
        check=check_obj,
    )
    registered_checks[check_data["name"]] = result.id
    print(f"  ✓ Registered: {check_data['name']} (ID: {result.id})")

print(f"\nTotal checks registered: {len(registered_checks)}")


Registering check: LLM09-citation-fabrication-detector from citation-fabrication-detector.md
  ✓ Registered: LLM09-citation-fabrication-detector (ID: 5d7a84ef-4d09-49f5-80fc-cb37d596956e)
Registering check: LLM09-factual-grounding-detector from factual-grounding-detector.md
  ✓ Registered: LLM09-factual-grounding-detector (ID: b8d6b91e-a09d-4151-8420-e8c98c6638e3)
Registering check: LLM09-high-stakes-harm-detector from high-stakes-harm-detector.md
  ✓ Registered: LLM09-high-stakes-harm-detector (ID: f4b4422b-18c0-4e6d-8001-75e370757ed3)

Total checks registered: 3


### Artifact Upload Summary

In [5]:
print("=" * 60)
print("LLM09 Misinformation — Artifact Upload Summary")
print("=" * 60)
print(f"\nScenarios ({len(registered_scenarios)}):")
for name, sc in registered_scenarios.items():
    print(f"  • {name} → {sc.scenario_id}")
print(f"\nChecks ({len(registered_checks)}):")
for name, cid in registered_checks.items():
    print(f"  • {name} → {cid}")
print("\n✓ All artifacts ready. Proceeding to evaluation...")

LLM09 Misinformation — Artifact Upload Summary

Scenarios (3):
  • LLM09-fabricated-citation → b91d9406-19b8-46bd-b38a-85f0135f8c48
  • LLM09-factual-hallucination → 388c1749-ba7a-4d80-9963-63f1cd0b742c
  • LLM09-high-stakes-misleading → 49df5c4e-8bc5-4214-a58e-336e653ea5b0

Checks (3):
  • LLM09-citation-fabrication-detector → 5d7a84ef-4d09-49f5-80fc-cb37d596956e
  • LLM09-factual-grounding-detector → b8d6b91e-a09d-4151-8420-e8c98c6638e3
  • LLM09-high-stakes-harm-detector → f4b4422b-18c0-4e6d-8001-75e370757ed3

✓ All artifacts ready. Proceeding to evaluation...


---
## Part 2 — Run Evaluation

### Configuration

The target agent is loaded from the shared `owasp/target.env` file (copy `owasp/target.env.example` and fill in your values).
All OWASP category notebooks reference this same file so that every control evaluates the same agent.

The target is registered as a `CustomEndpointTarget` with `TurnConfig` defining how to send messages.
All LLM09 scenarios are single-turn (`max_turns=1`) — no multi-turn simulations or adversarial drivers are needed.

In [6]:
# Target loaded from owasp/target.env. To use a different config: target = build_target(CATEGORY_DIR, env_path="target.prod.env")
target = build_target(CATEGORY_DIR)
TARGET_NAME = target.name
print(f"\u2713 Target agent: {TARGET_NAME}")

SCENARIO_CHECK_PAIRS = {
    "LLM09-factual-hallucination": "LLM09-factual-grounding-detector",
    "LLM09-fabricated-citation": "LLM09-citation-fabrication-detector",
    "LLM09-high-stakes-misleading": "LLM09-high-stakes-harm-detector",
}

✓ Target agent: FinanceBot


### Build Target

Constructs a `CustomEndpointTarget` from `owasp/target.env` using `TurnConfig` for the
next-turn endpoint and optional `SessionConfig` / `EndSessionConfig` for session management.

In [7]:
# Target built in config cell above via build_target(CATEGORY_DIR)


### Single-Turn Tests — All 3 Scenarios

Each scenario runs via `okareo.run_simulation()` constrained to `max_turns=1`.
A pass-through driver delivers the scenario input verbatim to the target agent; the agent's
response is evaluated by the scenario's paired model-based check.

| Scenario | Check | Domain |
|----------|-------|--------|
| LLM09-factual-hallucination | LLM09-factual-grounding-detector | History, science, geography, current affairs |
| LLM09-fabricated-citation | LLM09-citation-fabrication-detector | Paper titles, DOIs, URLs, bibliographic details |
| LLM09-high-stakes-misleading | LLM09-high-stakes-harm-detector | Medical, legal, financial |

In [8]:
single_turn_driver = Driver(
    temperature=0,
    name=f"{TARGET_NAME}-single-turn-driver",
    prompt_template=SINGLE_TURN_DRIVER_TEMPLATE,
)

evaluation_results = {}

for scenario_name, check_name in SCENARIO_CHECK_PAIRS.items():
    print(f"\n{'='*60}")
    print(f"Running: {scenario_name}")
    print(f"Check:   {check_name}")
    print(f"{'='*60}")
    try:
        scenario = registered_scenarios[scenario_name]

        test_run = okareo.run_simulation(
            target=target,
            driver=single_turn_driver,
            name=f"LLM09 Eval — {scenario_name}",
            api_key=OKAREO_API_KEY,
            first_turn="driver",
            scenario=scenario,
            max_turns=1,
            checks=[check_name],
        )
        evaluation_results[scenario_name] = test_run
        print(f"  ✓ Test run complete: {test_run.id}")
        if hasattr(test_run, "app_link") and test_run.app_link:
            print(f"  View: {test_run.app_link}")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        evaluation_results[scenario_name] = None


Running: LLM09-factual-hallucination
Check:   LLM09-factual-grounding-detector
  ✓ Test run complete: 31116c6b-19ac-4d07-ba6e-bc90460c7b6e
  View: https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/31116c6b-19ac-4d07-ba6e-bc90460c7b6e

Running: LLM09-fabricated-citation
Check:   LLM09-citation-fabrication-detector
  ✓ Test run complete: 34dbdac2-c5b5-4790-8110-eccd59713e81
  View: https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/34dbdac2-c5b5-4790-8110-eccd59713e81

Running: LLM09-high-stakes-misleading
Check:   LLM09-high-stakes-harm-detector
  ✓ Test run complete: c7a8cc08-b6f5-4a89-980e-1c4a2d86de2a
  View: https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/c7a8cc08-b6f5-4a89-980e-1c4a2d86de2a


### Results Summary

In [11]:
print("\n" + "=" * 60)
print("LLM09 MISINFORMATION — EVALUATION RESULTS")
print("OWASP Category: LLM09 | Risk Severity: Medium")
print("=" * 60)

print(f"\n{'Scenario':<42} {'Check':<42} {'Status':<10} {'Link / Run ID'}")
print("-" * 140)
for scenario_name, result in evaluation_results.items():
    check_name = SCENARIO_CHECK_PAIRS.get(scenario_name, "N/A")
    if result is None:
        print(f"{scenario_name:<42} {check_name:<42} {'ERROR':<10} N/A")
    else:
        link = getattr(result, "app_link", None) or result.id
        print(f"{scenario_name:<42} {check_name:<42} {'COMPLETE':<10} {link}")

errors = sum(1 for r in evaluation_results.values() if r is None)
print(f"\nTotal evaluated: {len(evaluation_results)} | Errors: {errors}")
if not errors:
    print("✓ All scenarios completed. See Okareo dashboard for full results.")


LLM09 MISINFORMATION — EVALUATION RESULTS
OWASP Category: LLM09 | Risk Severity: Medium

Scenario                                   Check                                      Status     Link / Run ID
--------------------------------------------------------------------------------------------------------------------------------------------
LLM09-factual-hallucination                LLM09-factual-grounding-detector           COMPLETE   https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/31116c6b-19ac-4d07-ba6e-bc90460c7b6e
LLM09-fabricated-citation                  LLM09-citation-fabrication-detector        COMPLETE   https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/34dbdac2-c5b5-4790-8110-eccd59713e81
LLM09-high-stakes-misleading               LLM09-high-stakes-harm-detector            COMPLETE   https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/c7a8cc08-b6f5-4a89-980e-1c4a2d86de2a

Total evaluated: 3 | Errors: 0
✓ All

### Detailed Results (Optional)

Retrieve per-row scores and model outputs for any completed test run.

In [10]:
# Uncomment to inspect a specific completed run in detail:
# from okareo_api_client.models import TestRunItem
# RUN_ID = "<paste_run_id_here>"
# detailed = okareo.get_test_run(RUN_ID)
# for row in (detailed.model_results or []):
#     print(f"Input:   {str(row.get('scenario_input', ''))[:80]}")
#     print(f"Output:  {str(row.get('model_output', ''))[:80]}")
#     print(f"Checks:  {row.get('check_scores', {})}")
#     print("-" * 40)